# Urethanase-Boltz2
Run the UMG-SP2/A1D5H retrospective co-folding benchmark with Boltz-2. A GPU runtime is recommended.

**Before running:** in Colab choose **Runtime → Change runtime type → T4 GPU** (or another available NVIDIA GPU).

In [ ]:
!nvidia-smi
!git clone https://github.com/jjimenezgar/Urethanase-Boltz2.git
%cd Urethanase-Boltz2
!pip -q install -e . 'boltz[cuda]'
!boltz --help | head -n 5

In [ ]:
!python scripts/fetch_reference.py
!python scripts/build_boltz_input.py
!cat inputs/umg_sp2_a1d5h.yaml

## Boltz-2 inference
No experimental structure/template or binding-pocket constraint is supplied to Boltz. The X-ray structure remains evaluation-only.

In [ ]:
!boltz predict inputs/umg_sp2_a1d5h.yaml --use_msa_server --out_dir results/boltz2

In [ ]:
from pathlib import Path
predictions = sorted(Path('results/boltz2/predictions/umg_sp2_a1d5h').glob('*_model_0.cif'))
assert predictions, 'Boltz-2 prediction CIF not found; inspect the previous cell output.'
pred = str(predictions[0])
print('Evaluating:', pred)
!urethanase-boltz2 evaluate --reference data/reference/8XTC.cif --prediction $pred --ligand A1D5H --output results/metrics.json
!cat results/metrics.json